# FatFormer XLA — demo Colab GPU
Chọn **Runtime → Change runtime type → GPU**. Dùng cùng app/model với bản Windows.

Notebook này chưa được nghiệm thu trên GPU trong phiên triển khai Windows. Cell 2 tạo log kiểm chứng thực tế của runtime hiện tại; không dùng kết quả benchmark cũ thay thế.

Chuẩn bị `fatformer_xla_demo.zip` và SHA-256 từ bản Windows (`artifacts/demo/fatformer_xla_demo.sha256`). Bundle chứa source + 20 ảnh minh họa, không chứa model. Model đọc từ Drive ở cell 2. Chỉ dùng bundle bạn tin cậy; manifest kiểm tra toàn vẹn, không phải chữ ký số.


In [ ]:
# 1. Upload source bundle, kiểm hash, stage vào /content và cài UI.
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, io, json, subprocess, sys, zipfile, os

expected_zip_sha = input("Dán SHA-256 từ file .sha256 trên Windows: ").strip().lower()
if len(expected_zip_sha) != 64 or any(c not in "0123456789abcdef" for c in expected_zip_sha):
    raise ValueError("SHA-256 phải có 64 ký tự hex.")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Chỉ upload một file fatformer_xla_demo.zip.")
bundle = next(iter(uploaded.values()))
if hashlib.sha256(bundle).hexdigest() != expected_zip_sha:
    raise ValueError("Bundle không khớp SHA-256.")
root = Path("/content/fatformer_xla_demo")
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    names = archive.namelist()
    manifest = json.loads(archive.read("demo_bundle_manifest.json"))
    if len(names) != len(set(names)) or set(names) != set(manifest) | {"demo_bundle_manifest.json"}:
        raise ValueError("Danh sách file trong bundle không khớp manifest.")
    for name, digest in manifest.items():
        parts = PurePosixPath(name)
        if parts.is_absolute() or ".." in parts.parts or "\\" in name:
            raise ValueError("Đường dẫn bundle không an toàn.")
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != digest:
            raise ValueError(f"Sai hash: {name}")
        path = root / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(data)
del uploaded, bundle
os.chdir(root)
sys.path.insert(0, str(root))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-demo.txt"], check=True)
print("Source sẵn sàng:", root)


In [ ]:
# 2. Mount Drive, copy đúng model vào SSD runtime và kiểm chứng GPU.
from google.colab import drive
import shutil, torch
if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU. Đổi runtime sang GPU rồi chạy lại từ cell 1.")
drive.mount("/content/drive")
drive_checkpoint = Path("/content/drive/MyDrive/Fatformer/checkpoint/phase7/corrected_20261109_102954_3d28ad/unified_fatformer_v2.pth")
local_checkpoint = root / "DATASET/checkpoints/phase7/demo/unified_fatformer_v2.pth"
if not drive_checkpoint.is_file():
    raise FileNotFoundError(f"Không tìm thấy model: {drive_checkpoint}. Sửa đúng đường dẫn Drive của bạn.")
local_checkpoint.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(drive_checkpoint, local_checkpoint)
from demo import load_demo_model, verify_demo, build_ui, launch_demo
model, info = load_demo_model(local_checkpoint, device="cuda")
print(json.dumps(info, indent=2))
report_path = root / "logs/demo/cuda_verification.json"
report = verify_demo(model, info, report_path)
print(json.dumps({k:v for k,v in report.items() if k != "results"}, indent=2))
files.download(str(report_path))


In [ ]:
# 3. Mở giao diện. Mỗi phiên dùng thông tin đăng nhập mới, không lưu vào notebook.
import getpass
username = input("Tên đăng nhập phiên demo: ").strip()
password = getpass.getpass("Mật khẩu phiên demo: ")
if not username or not password:
    raise ValueError("Tên đăng nhập và mật khẩu không được rỗng.")
app = build_ui(model, info)
launch_demo(app, share=True, auth=(username, password), prevent_thread_lock=True)
del password


Dùng link `gradio.live`, đăng nhập, chọn ảnh hoặc ảnh mẫu rồi bấm **Phân tích ảnh**. Có thể chọn so sánh JPEG Q30. Link chỉ dùng khi runtime còn hoạt động; dừng runtime để kết thúc phiên. Không lưu notebook kèm output chứa link phiên hoặc thông tin riêng.

Muốn xem chi tiết, bật **Giải thích vùng ảnh**, bấm Phân tích rồi mở mục **Vùng ảnh ảnh hưởng đến điểm Fake**. App chạy thêm 16 lần che vùng trên crop gốc. Bản đồ thể hiện độ nhạy của điểm Fake, chưa xác nhận vùng nào là giả mạo; latency và tính năng này trên Colab GPU cần được kiểm chứng thực tế.

Nếu cần chạy lại sau khi model đã nạp, chỉ chạy cell 3 (đóng app cũ bằng `app.close()` trước). Nếu thiếu VRAM, khởi động lại runtime và chạy lại từ đầu. Tải log GPU về để đối chiếu cùng backend; CPU FP32 và GPU FP16 không được mặc định bit-exact.
